In [1]:
import pandas as pd

customers = pd.read_csv("customers_cleaned.csv", parse_dates=["date_of_birth", "acquisition_date", "churn_date"])
subscriptions = pd.read_csv("subscriptions_cleaned.csv", parse_dates=["start_date", "end_date"])
plans = pd.read_csv("plans_cleaned.csv")
states = pd.read_csv("states_cleaned.csv")
regions = pd.read_csv("regions_cleaned.csv")
network_quality = pd.read_csv("network_quality_cleaned.csv")
complaints = pd.read_csv("complaints_cleaned.csv", parse_dates=["complaint_date", "resolution_date"])

dash = customers[["customer_id", "customer_segment", "product_line", "city_tier", "city_name", "state_id", "region_id",
                  "occupation", "gender", "age", "tenure_months", "customer_status", "acquisition_date", "churn_date", "plan_id"]].copy()
dash = dash.merge(states[["state_id", "state_name"]], on="state_id", how="left")
dash = dash.merge(regions[["region_id", "region_name"]], on="region_id", how="left")
dash = dash.merge(plans[["plan_id", "billing_cycle"]], on="plan_id", how="left")

# monthly charges of each customer
all_charge = subscriptions.groupby("customer_id")["monthly_charge_inr"].sum().reset_index(name="monthly_charge")
active_charge = subscriptions[subscriptions["subscription_status"] == "Active"].groupby("customer_id")["monthly_charge_inr"].sum().reset_index(name="active_charge")
dash = dash.merge(all_charge, on="customer_id", how="left")
dash = dash.merge(active_charge, on="customer_id", how="left")
dash["active_charge"] = dash["active_charge"].fillna(0)
dash["current_mrr"] = dash["active_charge"].where(dash["customer_status"] == "Active", 0)
dash["churned"] = (dash["customer_status"] == "Churned").astype(int)

# tenure band (numbers in front so Power BI sorts them correctly)
dash["tenure_band"] = pd.cut(dash["tenure_months"], bins=[-1, 3, 12, 36, 1000],
                             labels=["1. 0-3 months", "2. 3-12 months", "3. 1-3 years", "4. 3+ years"]).astype(str)

# unresolved complaints
unresolved = complaints[complaints["status"] == "Unresolved"].groupby("customer_id").size().reset_index(name="unresolved_complaints")
dash = dash.merge(unresolved, on="customer_id", how="left")
dash["unresolved_complaints"] = dash["unresolved_complaints"].fillna(0).astype(int)
dash["complaint_group"] = dash["unresolved_complaints"].clip(upper=3).astype(str).replace({"3": "3+"})

# network quality
network = network_quality.groupby("customer_id")[["dropped_call_rate_pct", "avg_download_speed_mbps", "avg_signal_strength_dbm", "network_downtime_hours"]].mean().round(2).reset_index()
network = network.rename(columns={"dropped_call_rate_pct": "avg_dropped_call_rate"})
dash = dash.merge(network, on="customer_id", how="left")
dash["call_group"] = pd.cut(dash["avg_dropped_call_rate"], bins=[0, 1.5, 2.5, 3.5, 100],
                            labels=["1. Low (<1.5%)", "2. Medium (1.5-2.5%)", "3. High (2.5-3.5%)", "4. Very high (3.5%+)"]).astype(str)

# simple churn-risk score from what the EDA showed
dash["risk_score"] = ((dash["billing_cycle"] == "Monthly") * 2 + (dash["tenure_months"] <= 12) * 2
                      + (dash["unresolved_complaints"] >= 2) * 2 + (dash["unresolved_complaints"] == 1) * 1
                      + (dash["avg_dropped_call_rate"] >= 2.5) * 1 + dash["customer_segment"].isin(["Mass", "Value"]) * 1)
dash["risk_level"] = pd.cut(dash["risk_score"], bins=[-1, 2, 4, 10], labels=["Low", "Medium", "High"]).astype(str)

dash.to_csv("dash_customers.csv", index=False)

print("dash_customers:", dash.shape)
print(dash["risk_level"].value_counts().to_dict())
active = dash[dash["customer_status"] == "Active"]
print("Active customers by risk:", active["risk_level"].value_counts().to_dict())
print("Churn rate by risk level:", dash.groupby("risk_level")["churned"].mean().mul(100).round(2).to_dict())
print("Revenue at risk (High-risk active customers): Rs", round(active[active["risk_level"] == "High"]["current_mrr"].sum()))

dash_customers: (19000, 32)
{'Medium': 9329, 'Low': 6670, 'High': 3001}
Active customers by risk: {'Medium': 7668, 'Low': 6291, 'High': 1535}
Churn rate by risk level: {'High': 48.85, 'Low': 5.68, 'Medium': 17.8}
Revenue at risk (High-risk active customers): Rs 892276


### Monthly table and the other three tables

In [2]:
support_tickets = pd.read_csv("support_tickets_cleaned.csv", parse_dates=["created_date", "resolution_date"])
retention = pd.read_csv("retention_campaigns_cleaned.csv", parse_dates=["contact_date"])

# ---------- monthly table (36 months: Apr 2023 to Mar 2026)
rows = []
for m in pd.period_range("2023-04", "2026-03", freq="M"):
    month_start = m.start_time
    month_end = m.end_time.normalize()

    start = ((customers["acquisition_date"] < month_start) & (customers["churn_date"].isnull() | (customers["churn_date"] >= month_start))).sum()
    new = ((customers["acquisition_date"] >= month_start) & (customers["acquisition_date"] <= month_end)).sum()
    lost = ((customers["churn_date"] >= month_start) & (customers["churn_date"] <= month_end)).sum()
    end = start + new - lost

    # MRR at the end of the month: charges of subscriptions that were live for customers who were still active
    active_ids = customers[(customers["acquisition_date"] <= month_end) & (customers["churn_date"].isnull() | (customers["churn_date"] > month_end))]["customer_id"]
    live_subs = subscriptions[(subscriptions["start_date"] <= month_end) & (subscriptions["end_date"].isnull() | (subscriptions["end_date"] > month_end)) & (subscriptions["customer_id"].isin(active_ids))]
    mrr = live_subs["monthly_charge_inr"].sum()

    # monthly revenue lost: charges of customers who churned in this month
    lost_ids = customers[(customers["churn_date"] >= month_start) & (customers["churn_date"] <= month_end)]["customer_id"]
    revenue_lost = dash[dash["customer_id"].isin(lost_ids)]["monthly_charge"].sum()

    rows.append([month_start.date(), start, new, lost, end, mrr, revenue_lost])

monthly = pd.DataFrame(rows, columns=["month", "customers_at_start", "new_customers", "lost_customers", "customers_at_end", "mrr", "revenue_lost"])
monthly["churn_rate_pct"] = (monthly["lost_customers"] / monthly["customers_at_start"] * 100).round(2)
monthly["retention_rate_pct"] = ((monthly["customers_at_end"] - monthly["new_customers"]) / monthly["customers_at_start"] * 100).round(2)
monthly.to_csv("dash_monthly.csv", index=False)
print("dash_monthly:", monthly.shape)
print(monthly.tail(4))

# ---------- tickets, complaints and campaigns, with customer details added
info = dash[["customer_id", "customer_segment", "state_name", "region_name", "customer_status"]]

tickets = support_tickets.merge(info, on="customer_id", how="left")
tickets.to_csv("dash_tickets.csv", index=False)

complaints_dash = complaints.merge(info, on="customer_id", how="left")
complaints_dash.to_csv("dash_complaints.csv", index=False)

campaigns = retention.merge(info, on="customer_id", how="left")
campaigns.to_csv("dash_campaigns.csv", index=False)

print("dash_tickets:", tickets.shape)
print("dash_complaints:", complaints_dash.shape)
print("dash_campaigns:", campaigns.shape)

dash_monthly: (36, 9)
         month  customers_at_start  new_customers  lost_customers  \
32  2025-12-01               15755            106             109   
33  2026-01-01               15752            113             124   
34  2026-02-01               15741             90             143   
35  2026-03-01               15688              4             198   

    customers_at_end         mrr  revenue_lost  churn_rate_pct  \
32             15752  14854757.0       76240.0            0.69   
33             15741  14846159.0      100648.0            0.79   
34             15688  14803565.0       96375.0            0.91   
35             15494  14671843.0      140792.0            1.26   

    retention_rate_pct  
32               99.31  
33               99.21  
34               99.09  
35               98.74  
dash_tickets: (31490, 15)
dash_complaints: (16584, 11)
dash_campaigns: (6475, 12)
